In [1]:
# Cell 1 — Libraries + Data Load

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    average_precision_score
)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

plt.style.use('dark_background')

print("✅ Libraries imported!")

# Paths
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
DATA_PATH = os.path.join(BASE_DIR, 'data', 'model_features.csv')
MODELS_PATH = os.path.join(BASE_DIR, 'models')

# Data load karo
print("\nLoading model_features.csv...")
df = pd.read_csv(DATA_PATH)

print(f"✅ Data loaded: {df.shape}")
print(f"\nFraud distribution:")
print(df['isFraud'].value_counts())
print(f"\nFraud %: {df['isFraud'].mean()*100:.4f}%")
print(f"\nAny nulls: {df.isnull().sum().sum()}")

✅ Libraries imported!

Loading model_features.csv...
✅ Data loaded: (6362620, 31)

Fraud distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Fraud %: 0.1291%

Any nulls: 0


In [2]:
# Cell 2 — Train/Test Split

# Features aur target alag karo
# X = input features (jo model dekhega)
# y = target (jo model predict karega)

X = df.drop('isFraud', axis=1)
y = df['isFraud']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"Feature columns: {X.columns.tolist()}")

# Train test split
# test_size=0.2 → 20% test, 80% train
# random_state=42 → reproducible results
# stratify=y → fraud % same rahega dono mein
#   (bina stratify ke test mein fraud kam ho sakta tha)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # important for imbalanced data
)

print(f"\n✅ Split done!")
print(f"Train size: {X_train.shape[0]:,} ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"Test size: {X_test.shape[0]:,} ({X_test.shape[0]/len(df)*100:.1f}%)")
print(f"\nFraud in train: {y_train.sum():,} ({y_train.mean()*100:.4f}%)")
print(f"Fraud in test: {y_test.sum():,} ({y_test.mean()*100:.4f}%)")

Features (X): (6362620, 30)
Target (y): (6362620,)
Feature columns: ['hour', 'day_of_week', 'is_weekend', 'is_odd_hour', 'amount_inr', 'amount_log', 'amount_to_balance', 'balance_after_zero', 'oldbalance_inr', 'newbalance_inr', 'balance_mismatch', 'has_mismatch', 'is_structuring', 'is_ctr_threshold', 'is_rtgs_range', 'sender_txn_count', 'sender_total_amount', 'receiver_txn_count', 'receiver_total_amount', 'type_ATM_WITHDRAWAL', 'type_CASH_DEPOSIT', 'type_NACH', 'type_NEFT', 'type_UPI', 'acct_business', 'acct_current', 'acct_jan_dhan', 'acct_savings', 'acct_student', 'is_profile_mismatch']

✅ Split done!
Train size: 5,090,096 (80.0%)
Test size: 1,272,524 (20.0%)

Fraud in train: 6,570 (0.1291%)
Fraud in test: 1,643 (0.1291%)


In [3]:
# Cell 3 — SMOTE

print("=== SMOTE — Handling Class Imbalance ===\n")

print(f"Before SMOTE:")
print(f"Train fraud: {y_train.sum():,} ({y_train.mean()*100:.4f}%)")
print(f"Train normal: {(y_train==0).sum():,}")

# SMOTE — Synthetic Minority Oversampling
# sampling_strategy=0.1 matlab —
# fraud ko normal ka 10% tak badhao
# 6,570 → ~5,09,009 (10% of normal)
# Itna kaafi hai — 50-50 nahi karte
# kyunki bahut zyada synthetic data = unrealistic

print("\nApplying SMOTE (may take 2-3 minutes)...")

smote = SMOTE(
    sampling_strategy=0.1,  # fraud = 10% of normal
    random_state=42,
    k_neighbors=5  # har synthetic example ke liye
                   # 5 nearest neighbors use karo
)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"\n✅ After SMOTE:")
print(f"Train size: {len(X_train_smote):,}")
print(f"Fraud: {y_train_smote.sum():,} ({y_train_smote.mean()*100:.2f}%)")
print(f"Normal: {(y_train_smote==0).sum():,}")
print(f"\nFraud examples added: {y_train_smote.sum() - y_train.sum():,}")

=== SMOTE — Handling Class Imbalance ===

Before SMOTE:
Train fraud: 6,570 (0.1291%)
Train normal: 5,083,526

Applying SMOTE (may take 2-3 minutes)...

✅ After SMOTE:
Train size: 5,591,878
Fraud: 508,352 (9.09%)
Normal: 5,083,526

Fraud examples added: 501,782


In [4]:
# Cell 4 — Isolation Forest

print("=== ISOLATION FOREST TRAINING ===\n")

# Isolation Forest —
# n_estimators = 100 trees banao
# contamination = expected fraud % 
#   0.001 = 0.1% — hamare dataset jaisa
# random_state = reproducible results
# n_jobs = -1 = saare CPU cores use karo (fast)

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.002,
    random_state=42,
    n_jobs=-1
)

print("Training Isolation Forest...")
print("(Using original train data — no SMOTE needed)")

# Sirf X_train use karo — y_train nahi
# Unsupervised = labels ki zaroorat nahi
iso_forest.fit(X_train)

print("✅ Isolation Forest trained!")

# Test set pe predict karo
# predict() returns: 1 = normal, -1 = anomaly
iso_predictions = iso_forest.predict(X_test)

# -1 ko 1 mein convert karo (fraud = 1)
# 1 ko 0 mein convert karo (normal = 0)
iso_fraud_pred = (iso_predictions == -1).astype(int)

# Score nikalo — lower score = more anomalous
iso_scores = iso_forest.decision_function(X_test)
# Negate karo — higher = more suspicious
iso_scores_normalized = -iso_scores

print(f"\n=== ISOLATION FOREST RESULTS ===")
print(f"Flagged as anomaly: {iso_fraud_pred.sum():,}")
print(f"Actual fraud in test: {y_test.sum():,}")
print(f"\nClassification Report:")
print(classification_report(y_test, iso_fraud_pred,
      target_names=['Normal', 'Fraud']))

# AUC-ROC
iso_auc = roc_auc_score(y_test, iso_scores_normalized)
print(f"AUC-ROC Score: {iso_auc:.4f}")

# Save model
joblib.dump(iso_forest,
    os.path.join(MODELS_PATH, 'isolation_forest_v1.pkl'))
print(f"\n✅ Model saved → models/isolation_forest_v1.pkl")


=== ISOLATION FOREST TRAINING ===

Training Isolation Forest...
(Using original train data — no SMOTE needed)
✅ Isolation Forest trained!

=== ISOLATION FOREST RESULTS ===
Flagged as anomaly: 2,605
Actual fraud in test: 1,643

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00   1270881
       Fraud       0.05      0.07      0.06      1643

    accuracy                           1.00   1272524
   macro avg       0.52      0.54      0.53   1272524
weighted avg       1.00      1.00      1.00   1272524

AUC-ROC Score: 0.9105

✅ Model saved → models/isolation_forest_v1.pkl


In [ ]:
# CELL 5
print("=== XGBOOST V2 — WITHOUT SMOTE ===\n")

# Class weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count

print(f"Scale weight: {scale_weight:.2f}")

# Model without SMOTE
xgb_model_v2 = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False
)

print("Training XGBoost WITHOUT SMOTE...")
print("(Original train data — real patterns only)\n")

# Original X_train, y_train — no SMOTE
xgb_model_v2.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)

print("\n✅ XGBoost V2 trained!")

# Predict
xgb_pred_v2 = xgb_model_v2.predict(X_test)
xgb_prob_v2 = xgb_model_v2.predict_proba(X_test)[:, 1]

print("\n=== RESULTS ===")
print(f"Flagged: {xgb_pred_v2.sum():,}")
print(f"Actual fraud: {y_test.sum():,}")
print(f"\nClassification Report:")
print(classification_report(
    y_test, xgb_pred_v2,
    target_names=['Normal', 'Fraud']
))

xgb_auc_v2 = roc_auc_score(y_test, xgb_prob_v2)
ap_v2 = average_precision_score(y_test, xgb_prob_v2)
print(f"AUC-ROC: {xgb_auc_v2:.4f}")
print(f"Average Precision: {ap_v2:.4f}")

# Save
joblib.dump(
    xgb_model_v2,
    os.path.join(MODELS_PATH, 'xgboost_v2.pkl')
)
print(f"\n✅ Saved → models/xgboost_v2.pkl")

=== XGBOOST V2 — WITHOUT SMOTE ===

Scale weight: 773.75
Training XGBoost WITHOUT SMOTE...
(Original train data — real patterns only)

[0]	validation_0-aucpr:0.99763
[100]	validation_0-aucpr:0.99761
[200]	validation_0-aucpr:0.99760
[300]	validation_0-aucpr:0.99761
[400]	validation_0-aucpr:0.99762
[499]	validation_0-aucpr:0.99763

✅ XGBoost V2 trained!

=== RESULTS ===
Flagged: 1,639
Actual fraud: 1,643

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00   1270881
       Fraud       1.00      1.00      1.00      1643

    accuracy                           1.00   1272524
   macro avg       1.00      1.00      1.00   1272524
weighted avg       1.00      1.00      1.00   1272524

AUC-ROC: 0.9998
Average Precision: 0.9976

✅ Saved → models/xgboost_v2.pkl


In [ ]:
# Cell 6 → XGBoost Exp3 train (SMOTE + dropped)

print("=== EXPERIMENT — type_UPI + has_mismatch DROP (SMOTE) ===\n")

# Dono drop karo
X_exp_both = X.drop(['type_UPI', 'has_mismatch'], axis=1)

print(f"Features remaining: {X_exp_both.shape[1]}")
print(f"Dropped: type_UPI, has_mismatch\n")

# Split
X_train_both, X_test_both, y_train_both, y_test_both = train_test_split(
    X_exp_both, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE
smote = SMOTE(sampling_strategy=0.1, random_state=42, k_neighbors=5)
X_train_both_sm, y_train_both_sm = smote.fit_resample(
    X_train_both, y_train_both
)

print(f"After SMOTE — Fraud: {y_train_both_sm.sum():,}")

# Model
xgb_both = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False
)

xgb_both.fit(
    X_train_both_sm, y_train_both_sm,
    eval_set=[(X_test_both, y_test_both)],
    verbose=100
)

pred_both = xgb_both.predict(X_test_both)
prob_both = xgb_both.predict_proba(X_test_both)[:, 1]

print(f"\nResults:")
print(classification_report(
    y_test_both, pred_both,
    target_names=['Normal', 'Fraud']
))
print(f"AUC-ROC: {roc_auc_score(y_test_both, prob_both):.4f}")
print(f"Average Precision: {average_precision_score(y_test_both, prob_both):.4f}")

imp_both = pd.DataFrame({
    'feature': X_exp_both.columns,
    'importance': xgb_both.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 10 features:")
print(imp_both.head(10))

=== EXPERIMENT — type_UPI + has_mismatch DROP (SMOTE) ===

Features remaining: 28
Dropped: type_UPI, has_mismatch

After SMOTE — Fraud: 508,352
[0]	validation_0-aucpr:0.99517
[100]	validation_0-aucpr:0.99780
[200]	validation_0-aucpr:0.99764
[300]	validation_0-aucpr:0.99764
[400]	validation_0-aucpr:0.99763
[499]	validation_0-aucpr:0.99762

Results:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00   1270881
       Fraud       0.98      1.00      0.99      1643

    accuracy                           1.00   1272524
   macro avg       0.99      1.00      0.99   1272524
weighted avg       1.00      1.00      1.00   1272524

AUC-ROC: 0.9996
Average Precision: 0.9976

Top 10 features:
               feature  importance
10    balance_mismatch    0.328282
7   balance_after_zero    0.105184
2           is_weekend    0.078020
6    amount_to_balance    0.075868
19   type_CASH_DEPOSIT    0.074919
4           amount_inr    0.071821
22       acct_busi

In [14]:
# CELL 7 — WEIGHTS CALCULATION 

print("=== CELL 7 — WEIGHTS CALCULATION ===\n")

# AUC scores jo already mile hain
auc_iso  = roc_auc_score(y_test, iso_scores_normalized)
auc_xgb2 = roc_auc_score(y_test, xgb_prob_v2)
auc_exp3 = roc_auc_score(y_test, prob_both)

print(f"AUC Scores —")
print(f"Isolation Forest : {auc_iso:.4f}")
print(f"XGBoost V2       : {auc_xgb2:.4f}")
print(f"XGBoost Exp3     : {auc_exp3:.4f}")

# Weights = AUC / Total AUC
total_auc = auc_iso + auc_xgb2 + auc_exp3

w1 = auc_iso  / total_auc
w2 = auc_xgb2 / total_auc
w3 = auc_exp3 / total_auc

print(f"\nWeights (AUC based) —")
print(f"Isolation Forest : {w1:.4f} ({w1*100:.1f}%)")
print(f"XGBoost V2       : {w2:.4f} ({w2*100:.1f}%)")
print(f"XGBoost Exp3     : {w3:.4f} ({w3*100:.1f}%)")
print(f"Total            : {w1+w2+w3:.4f} ✅")

=== CELL 7 — WEIGHTS CALCULATION ===

AUC Scores —
Isolation Forest : 0.9105
XGBoost V2       : 0.9998
XGBoost Exp3     : 0.9996

Weights (AUC based) —
Isolation Forest : 0.3129 (31.3%)
XGBoost V2       : 0.3436 (34.4%)
XGBoost Exp3     : 0.3435 (34.4%)
Total            : 1.0000 ✅


In [15]:
# CELL 8 — ENSEMBLE + THRESHOLD
print("=== CELL 8 — ENSEMBLE + THRESHOLD ===\n")

# Step 1 — ISO score normalize karo 0-1 mein
# Kyunki ISO score negative bhi hota hai
# XGB score already 0-1 mein hota hai
# Dono ko same scale pe laana zaroori hai

iso_min = iso_scores_normalized.min()
iso_max = iso_scores_normalized.max()
iso_scores_01 = (
    (iso_scores_normalized - iso_min) / 
    (iso_max - iso_min)
)

print(f"ISO score range after normalization —")
print(f"Min: {iso_scores_01.min():.4f}")
print(f"Max: {iso_scores_01.max():.4f}")

# Step 2 — Ensemble score banao
ensemble_score = (
    w1 * iso_scores_01 +
    w2 * xgb_prob_v2 +
    w3 * prob_both
)

print(f"\nEnsemble score range —")
print(f"Min: {ensemble_score.min():.4f}")
print(f"Max: {ensemble_score.max():.4f}")
print(f"Mean: {ensemble_score.mean():.4f}")

# Step 3 — Optimal threshold
from sklearn.metrics import precision_recall_curve

precision_vals, recall_vals, thresholds_pr = \
    precision_recall_curve(y_test, ensemble_score)

# F1 maximize karo
f1_vals = (
    2 * precision_vals * recall_vals / 
    (precision_vals + recall_vals + 1e-8)
)

optimal_idx = f1_vals.argmax()
optimal_threshold = thresholds_pr[optimal_idx]

print(f"\n=== OPTIMAL THRESHOLD ===")
print(f"Fraud Threshold  : {optimal_threshold:.4f}")
print(f"Review Threshold : {optimal_threshold*0.6:.4f}")
print(f"\nAt this threshold —")
print(f"Precision : {precision_vals[optimal_idx]:.4f}")
print(f"Recall    : {recall_vals[optimal_idx]:.4f}")
print(f"F1 Score  : {f1_vals[optimal_idx]:.4f}")

=== CELL 8 — ENSEMBLE + THRESHOLD ===

ISO score range after normalization —
Min: 0.0000
Max: 1.0000

Ensemble score range —
Min: 0.0000
Max: 0.9888
Mean: 0.0656

=== OPTIMAL THRESHOLD ===
Fraud Threshold  : 0.7262
Review Threshold : 0.4357

At this threshold —
Precision : 1.0000
Recall    : 0.9976
F1 Score  : 0.9988


In [16]:
# CELL 9 — FINAL EVALUATION
print("=== CELL 9 — FINAL EVALUATION ===\n")

# Final predictions
fraud_threshold  = optimal_threshold
review_threshold = optimal_threshold * 0.6

# Labels assign karo
def get_risk_label(score):
    if score >= fraud_threshold:
        return 'FRAUD'
    elif score >= review_threshold:
        return 'REVIEW'
    else:
        return 'NORMAL'

final_pred = (ensemble_score >= fraud_threshold).astype(int)

print("Classification Report —")
print(classification_report(
    y_test, final_pred,
    target_names=['Normal', 'Fraud']
))

print(f"AUC-ROC: {roc_auc_score(y_test, ensemble_score):.4f}")

# Risk level distribution
risk_labels = [get_risk_label(s) for s in ensemble_score]
from collections import Counter
dist = Counter(risk_labels)

print(f"\nRisk Level Distribution —")
print(f"FRAUD  🚨 : {dist['FRAUD']:,}")
print(f"REVIEW ⚠️  : {dist['REVIEW']:,}")
print(f"NORMAL ✅  : {dist['NORMAL']:,}")

# Fraud mein risk levels
fraud_scores = ensemble_score[y_test==1]
fraud_labels = [get_risk_label(s) for s in fraud_scores]
fraud_dist = Counter(fraud_labels)

print(f"\nActual Fraud mein Risk Levels —")
print(f"FRAUD  🚨 : {fraud_dist['FRAUD']:,}")
print(f"REVIEW ⚠️  : {fraud_dist['REVIEW']:,}")
print(f"NORMAL ✅  : {fraud_dist['NORMAL']:,}")

=== CELL 9 — FINAL EVALUATION ===

Classification Report —
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00   1270881
       Fraud       1.00      1.00      1.00      1643

    accuracy                           1.00   1272524
   macro avg       1.00      1.00      1.00   1272524
weighted avg       1.00      1.00      1.00   1272524

AUC-ROC: 0.9989

Risk Level Distribution —
FRAUD  🚨 : 1,639
REVIEW ⚠️  : 3
NORMAL ✅  : 1,270,882

Actual Fraud mein Risk Levels —
FRAUD  🚨 : 1,639
REVIEW ⚠️  : 0
NORMAL ✅  : 4


In [17]:
# Cell 10 — Save Everything
import json
print("=== CELL 10 — SAVE ALL ===\n")

# Models save karo
joblib.dump(iso_forest,
    os.path.join(MODELS_PATH, 'isolation_forest_final.pkl'))
print("✅ isolation_forest_final.pkl")

joblib.dump(xgb_model_v2,
    os.path.join(MODELS_PATH, 'xgboost_v2_final.pkl'))
print("✅ xgboost_v2_final.pkl")

joblib.dump(xgb_both,
    os.path.join(MODELS_PATH, 'xgboost_exp3_final.pkl'))
print("✅ xgboost_exp3_final.pkl")

# Config save karo
# Yeh Flask mein directly use hoga
config = {
    'weights': {
        'isolation_forest': round(w1, 4),
        'xgboost_v2'      : round(w2, 4),
        'xgboost_exp3'    : round(w3, 4)
    },
    'thresholds': {
        'fraud' : round(float(fraud_threshold), 4),
        'review': round(float(review_threshold), 4)
    },
    'features': {
        'xgboost_v2' : X_train.columns.tolist(),
        'xgboost_exp3': X_exp_both.columns.tolist()
    },
    'iso_normalization': {
        'min': round(float(iso_min), 4),
        'max': round(float(iso_max), 4)
    }
}

with open(os.path.join(MODELS_PATH, 'ensemble_config.json'), 'w') as f:
    json.dump(config, f, indent=2)
print("✅ ensemble_config.json")

print("\n=== SAVED FILES ===")
print("models/")
print("├── isolation_forest_final.pkl")
print("├── xgboost_v2_final.pkl")
print("├── xgboost_exp3_final.pkl")
print("└── ensemble_config.json")

print("\n=== TRAINING COMPLETE ===")
print(f"3 models trained ✅")
print(f"Weights calculated ✅")
print(f"Optimal threshold found ✅")
print(f"Config saved ✅")
print(f"Ready for Flask! 🔥")

=== CELL 10 — SAVE ALL ===

✅ isolation_forest_final.pkl
✅ xgboost_v2_final.pkl
✅ xgboost_exp3_final.pkl
✅ ensemble_config.json

=== SAVED FILES ===
models/
├── isolation_forest_final.pkl
├── xgboost_v2_final.pkl
├── xgboost_exp3_final.pkl
└── ensemble_config.json

=== TRAINING COMPLETE ===
3 models trained ✅
Weights calculated ✅
Optimal threshold found ✅
Config saved ✅
Ready for Flask! 🔥
